# DBLP Recovery：30k TEMP 数据集部署（Drive 持久化 / Colab 本地计算）

本 Notebook 专用于 `papers_master_temp.csv`（29,963 篇论文）。它使用独立的 `dblp_cache_30000_temp` 和 `recovery_output_30000_temp`，不会覆盖 599 篇旧结果。

**I/O 策略：** 启动时把 XML、DTD、Profile SQLite、Fetch 主表/作者表及 DBLP 辅助 CSV 分块复制到 `/content`；SQLite、XML 扫描和输入校验均在本地磁盘进行。新版 `bootstrap.py` 对 2GB 级 author-detail 与大型 conflict/priority audit 使用分块、窄列读取，避免将整张宽表及其字典副本一次性载入 RAM。Drive 只保留原始文件、阶段结束后的冷 Cache Snapshot，以及最终输出/日志。首次复制数 GB 文件会较久，但有字节进度与 SHA256 校验；同一 VM 重跑会复用本地副本。

先运行 PREPARE 单元完成本地复制，并检查可用 RAM 与大型 DBLP 辅助 CSV 表头；PREPARE 不启动 Recovery。确认 staging 成功后，可使用本版本的内存有界 Bootstrap 执行 `stage="scan"`。Stage 1 与 Stage 2 必须使用同一代码 ZIP。待冷 Cache Snapshot 成功写回 Drive 后，再执行 `stage="all"`。正常中断会等待 Python 关闭 SQLite 后备份；直接重置/回收 VM 无法保证尚未备份的本地进度。

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, subprocess, sys, shutil

uploaded = files.upload()
archive = next(name for name in uploaded if name.lower().endswith(".zip"))
code_root = Path("/content/GP_DBLP_Recovery_Code_30000_TEMP")
if code_root.exists():
    shutil.rmtree(code_root)
code_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (code_root / name).resolve().is_relative_to(code_root.resolve()):
            raise ValueError("Unsafe ZIP path")
    z.extractall(code_root)

project = code_root / "GP_DBLP_Author_Recovery"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-r",
    str(project / "requirements.txt")
])
print("Code package:", project)


In [ ]:
from pathlib import Path
import json, shutil

# Persistent Google Drive locations
drive_root = Path("/content/drive/MyDrive/GP_Research_Papers")
processed = drive_root / "data/processed"
dblp_dir = processed / "dblp_author_features_v3_1_1"
xml_path = drive_root / "data/raw/dblp/dblp.xml.gz"
dtd_path = drive_root / "data/raw/dblp/dblp.dtd"

# Dedicated 30k TEMP cache/output: never reuse the 599-paper directories.
cache_root = Path("/content/drive/MyDrive/dblp_cache_30000_temp")
output_root = Path("/content/drive/MyDrive/GP_DBLP_Author_Recovery/recovery_output_30000_temp")
master_path = processed / "papers_master_temp.csv"
authors_path = processed / "authors_arxiv_v1.csv"
index_path = cache_root / "index.sqlite"

required = [master_path, authors_path, xml_path, dtd_path, index_path]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

# The optional DBLP files are consumed by bootstrap.py when present.
optional_names = [
    "dblp_author_features.csv",
    "dblp_author_features_by_author.csv",
    "dblp_candidate_match_audit.csv",
    "dblp_identity_conflicts.csv",
    "dblp_identity_priority_audit.csv",
    "dblp_author_match_audit.csv",
    "dblp_temporal_exclusion_audit.csv",
    "dblp_source_metadata.json",
]
optional_files = [dblp_dir / name for name in optional_names if (dblp_dir / name).is_file()]

copy_bytes = sum(p.stat().st_size for p in [master_path, authors_path, xml_path, dtd_path, index_path] + optional_files)
free_bytes = shutil.disk_usage("/content").free
print("Required files:")
for p in [master_path, authors_path, xml_path, dtd_path, index_path] + optional_files:
    print(f"  {p.name}: {p.stat().st_size / 1024**3:.3f} GiB")
print(f"\nApproximate first-run local staging: {copy_bytes / 1024**3:.2f} GiB")
print(f"Available /content disk: {free_bytes / 1024**3:.2f} GiB")
if free_bytes < copy_bytes + 2 * 1024**3:
    raise OSError("Insufficient local disk space for staging plus 2 GiB safety reserve.")

config = json.loads((project / "config.colab.example.json").read_text())
config.update(
    FETCH_DATA_ROOT=str(drive_root),
    DBLP_XML_PATH=str(xml_path),
    DTD_PATH=str(dtd_path),
    CACHE_DIR=str(cache_root),
    PROFILE_INDEX_PATH=None,
    DERIVED_CACHE_DIR=None,
    OUTPUT_DIR=str(output_root),
    master_file="data/processed/papers_master_temp.csv",
    authors_file="data/processed/authors_arxiv_v1.csv",
    dblp_directory="data/processed/dblp_author_features_v3_1_1",
    resume_enabled=True,
    log_level="INFO",
)
config_path = project / "config.runtime.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("\nConfiguration written:", config_path)
print("Persistent Cache:", cache_root)
print("Persistent Output:", output_root)
print("Master:", master_path)


In [ ]:
import sys, shutil, json
import pandas as pd
sys.path.insert(0, str(project.resolve()))
from colab_runtime import prepare, launch

local_root = "/content/GP_DBLP_Recovery_Work_30000_TEMP"
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"Free local disk before staging: {free_gib:.2f} GiB")

# PREPARE ONLY: copy/verify inputs and restore SQLite locally, but do NOT start recovery.
effective_config_path, local_cache, persistent_cache = prepare(config, local_root)
runtime_config = json.loads(Path(effective_config_path).read_text())
local_fetch_root = Path(runtime_config["FETCH_DATA_ROOT"])

print("\nLocal staging completed.")
print("Local Fetch root:", local_fetch_root)
print("Local Cache:", local_cache)
print("Local XML:", runtime_config["DBLP_XML_PATH"])
print("Free local disk after staging:", round(shutil.disk_usage("/content").free / 1024**3, 2), "GiB")

# Header-only inspection: does not load the large CSV bodies into RAM.
inspect_rel = [
    runtime_config["master_file"],
    runtime_config["authors_file"],
    *[str(Path(runtime_config["dblp_directory"]) / name) for name in [
        "dblp_author_features.csv",
        "dblp_author_features_by_author.csv",
        "dblp_candidate_match_audit.csv",
        "dblp_identity_conflicts.csv",
        "dblp_identity_priority_audit.csv",
        "dblp_author_match_audit.csv",
        "dblp_temporal_exclusion_audit.csv",
        "dblp_source_metadata.json",
    ]],
]
for rel in inspect_rel:
    path = local_fetch_root / rel
    if not path.is_file() or path.suffix.lower() != ".csv":
        continue
    columns = pd.read_csv(path, nrows=0, dtype=str).columns.tolist()
    print(f"{path.name}: {len(columns)} columns; size={path.stat().st_size / 1024**3:.3f} GiB")

try:
    import psutil
    print("Available RAM:", round(psutil.virtual_memory().available / 1024**3, 2), "GiB")
except Exception:
    print("RAM telemetry unavailable; continue only if Colab shows sufficient RAM.")

print("\nBootstrap memory policy: large author-detail/conflict/audit CSVs are streamed in chunks; only required columns and roster-matched rows are retained.")
print("No Recovery stage has started yet. Review these sizes/headers before running Stage 1.")


In [ ]:
# Stage 1: full XML scan + disk dedup. On exit, closed SQLite files are
# checksum-verified and snapshotted to dblp_cache_30000_temp/runtime_snapshots.
effective_config_path = launch(
    project=project,
    config_path=config_path,
    local_root=local_root,
    stage="scan",
)
print("\nStage scan completed. Persistent cache snapshot should now be saved.")


In [ ]:
# Stage 2: network scoring, identity recovery, feature calculation,
# final audits, and Missing Treatment. Reuses the local cache from Stage 1.
effective_config_path = launch(
    project=project,
    config_path=config_path,
    local_root=local_root,
    stage="all",
)
print("\nStage all completed; Missing Treatment is included automatically.")


In [ ]:
from pathlib import Path
import pandas as pd, json

summary_path = output_root / "report/final_recovery_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print("No final summary yet. PREPARED/LOCAL_SCAN_COMPLETE is not final recovery completion.")

features_path = output_root / "features/final_team_features.csv"
if features_path.exists():
    display(pd.read_csv(features_path, nrows=5))

print("\nOutput directory:", output_root)


## 可选：单独重跑 Missing Treatment

正常执行 `stage="all"` 已自动运行 Missing Treatment，因此通常不需要执行下面的 Cell。只有 Recovery 已完整完成、但你想单独重跑 Treatment 时才使用。它必须读取同一份 `effective_config_path`，因为 source manifest 记录的是本次本地 staging 的输入路径；不要重新从 `config.colab.example.json` 生成 Drive 路径配置。

In [ ]:
import json, sys, subprocess
from pathlib import Path

if "effective_config_path" not in globals() or not Path(effective_config_path).is_file():
    raise RuntimeError("Run Stage scan/all first in this Colab runtime, or rerun stage=all to restore the workspace.")

treatment_config = json.loads(Path(effective_config_path).read_text())
treatment_config_path = Path(effective_config_path).with_name("config.treatment.runtime.json")
treatment_config_path.write_text(json.dumps(treatment_config, indent=2), encoding="utf-8")
subprocess.run([
    sys.executable, "-u", str(project / "main.py"),
    "--config", str(treatment_config_path), "--stage", "treatment",
], cwd=project, check=True)
print((Path(treatment_config["OUTPUT_DIR"]) / "report/missing_treatment_summary.md").read_text())
